[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/healthcareaibook/blob/main/vol%201%20notebooks/chapter_07/notebook_7_4_ecg_classification_1d_cnn.ipynb)

*Click the badge above to open this notebook in Google Colab (no setup required)*

---


# Notebook 7.4: ECG Signal Classification for AFib Detection

**Chapter 7: Time Series Analysis - Implementing Journey 2 (Yuki)**

**Journey Connection**: This notebook implements Journey 2 from Chapter 3, where Yuki received multiple false AFib alerts from a wearable device. For the clinical context and patient story, refer to Chapter 3.

## Learning Objectives

1. Preprocess ECG waveforms (filtering, quality assessment)
2. Extract hand-crafted features (R-R intervals, HRV)
3. Train a rhythm classifier two ways: a Random Forest on HRV features (runs everywhere, including the browser) and a 1D CNN on the raw ECG (needs TensorFlow), and compare accuracy, interpretability and computational cost
4. Analyze false positives and motion artifacts
5. Understand the impact of prevalence on PPV

## Clinical Context

Atrial fibrillation increases stroke risk 5×. Wearable ECG devices can catch paroxysmal AFib missed in office visits.

**Yuki's story** (Chapter 3, Journey 2): Yuki Tanaka, 55, a healthy marketing executive who exercises regularly, bought an ECG smartwatch after her mother's stroke from undiagnosed AFib. During a 6:12 AM jog it alerted: *"Possible atrial fibrillation."* A 12-lead ECG and a 48-hour Holter were normal: probably a false alarm from motion artifact. The alerts kept coming, and she eventually stopped wearing the watch continuously.

**The challenge**: In a 55-year-old without known heart disease, AFib prevalence is only ~0.1–0.5%. At 0.5% (Chapter 3), a device with 95% sensitivity and 98% specificity has a PPV of only ~19%: about **4 false alarms for every true AFib detection**.

---

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.stats import kurtosis, skew
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print("Libraries imported successfully!")

Libraries imported successfully!


## 1. Generate Synthetic ECG Data

We'll create **synthetic** ECG waveforms for normal sinus rhythm and atrial fibrillation. Both rhythms use the same QRS-T template and the same noise, so they differ mainly in **rhythm** (regular vs irregularly irregular R-R intervals), plus P waves (normal) vs small fibrillatory f-waves (AFib).

- Normal sinus rhythm: 60–90 bpm, R-R standard deviation ~0.02 s (mild sinus arrhythmia)
- AFib: 80–110 bpm on average, R-R standard deviation 0.15–0.20 s, intervals truncated at ≥ 0.3 s (≤ 200 bpm)

In [2]:
def ecg_beat_template(fs=250, p_wave=True):
    """One PQRST complex (0.6 s); the R peak sits 0.20 s after the beat start."""
    tb = np.arange(int(0.6 * fs)) / fs
    beat = (1.0 * np.exp(-((tb - 0.200) / 0.012) ** 2)      # R
            - 0.10 * np.exp(-((tb - 0.180) / 0.010) ** 2)   # Q
            - 0.25 * np.exp(-((tb - 0.225) / 0.010) ** 2)   # S
            + 0.30 * np.exp(-((tb - 0.400) / 0.050) ** 2))  # T
    if p_wave:
        beat += 0.12 * np.exp(-((tb - 0.080) / 0.025) ** 2)  # P
    return beat


def generate_synthetic_ecg(duration=30, fs=250, rhythm='normal'):
    """
    Generate a synthetic single-lead ECG waveform.

    Parameters:
    - duration: seconds
    - fs: sampling frequency (Hz)
    - rhythm: 'normal' or 'afib'

    Both rhythms share the same QRS-T template and noise level, so the
    classes differ by rhythm (R-R regularity), not by QRS amplitude.
    """
    t = np.arange(0, duration, 1/fs)
    n_beats = int(duration / 0.3) + 2

    if rhythm == 'normal':
        # Normal sinus rhythm: regular R-R intervals with mild sinus arrhythmia
        hr = np.random.uniform(60, 90)
        rr_intervals = np.random.normal(60 / hr, 0.02, n_beats)
    else:  # afib
        # AFib: irregularly irregular R-R intervals (SD 0.15-0.20 s), no P waves
        hr_mean = np.random.uniform(80, 110)
        rr_sd = np.random.uniform(0.15, 0.20)
        rr_intervals = np.random.normal(60 / hr_mean, rr_sd, n_beats)
        too_short = rr_intervals < 0.3          # truncate at 0.3 s (200 bpm):
        while too_short.any():                  # redraw the impossible ones
            rr_intervals[too_short] = np.random.normal(60 / hr_mean, rr_sd, too_short.sum())
            too_short = rr_intervals < 0.3

    beat = ecg_beat_template(fs, p_wave=(rhythm == 'normal'))
    ecg = np.zeros(len(t) + len(beat))
    beat_time = np.random.uniform(0, rr_intervals[0])  # random phase
    for rr in rr_intervals:
        idx = int(beat_time * fs)
        if idx >= len(t):
            break
        ecg[idx:idx + len(beat)] += beat
        beat_time += rr
    ecg = ecg[:len(t)]

    if rhythm == 'afib':
        # Fibrillatory f-waves: small 4-8 Hz baseline oscillation
        f = np.random.uniform(4, 8)
        ecg += 0.04 * np.sin(2 * np.pi * f * t + np.random.uniform(0, 2 * np.pi))

    # Add noise
    ecg += np.random.normal(0, 0.05, len(ecg))

    return t, ecg

# Generate examples
t_normal, ecg_normal = generate_synthetic_ecg(duration=10, rhythm='normal')
t_afib, ecg_afib = generate_synthetic_ecg(duration=10, rhythm='afib')

# Plot
fig, axes = plt.subplots(2, 1, figsize=(14, 8))

axes[0].plot(t_normal, ecg_normal, 'b-', linewidth=1)
axes[0].set_title('Normal Sinus Rhythm (synthetic)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Amplitude (mV)')
axes[0].set_xlim([0, 10])
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_afib, ecg_afib, 'r-', linewidth=1)
axes[1].set_title('Atrial Fibrillation (synthetic)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Time (seconds)')
axes[1].set_ylabel('Amplitude (mV)')
axes[1].set_xlim([0, 10])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Peak amplitude: normal {ecg_normal.max():.2f} mV vs AFib {ecg_afib.max():.2f} mV (same QRS template)")
print("Key differences:")
print("  Normal: Regular R-R intervals, visible P waves")
print("  AFib: Irregular R-R intervals, absent P waves, small fibrillatory baseline waves")


Key differences:
  Normal: Regular R-R intervals, visible P waves
  AFib: Irregular R-R intervals, absent P waves, irregular baseline


## 2. Feature-Based Classification

Traditional approach: Extract hand-crafted features from ECG signal.

In [3]:
def extract_rr_intervals(ecg, fs=250):
    """Detect R-peaks and calculate R-R intervals."""
    # Simple peak detection: half the height of the tallest R waves,
    # refractory period 0.25 s (allows heart rates up to 240 bpm)
    threshold = 0.5 * np.percentile(ecg, 99.5)
    peaks, _ = signal.find_peaks(ecg, height=threshold, distance=int(fs * 0.25))

    rr_intervals = np.diff(peaks) / fs  # Convert to seconds
    return peaks, rr_intervals

def extract_hrv_features(rr_intervals):
    """Calculate heart rate variability features."""
    if len(rr_intervals) < 2:
        return {}

    features = {
        'rr_mean': np.mean(rr_intervals),
        'rr_std': np.std(rr_intervals),
        'rr_rmssd': np.sqrt(np.mean(np.diff(rr_intervals)**2)),
        'rr_min': np.min(rr_intervals),
        'rr_max': np.max(rr_intervals),
        'rr_range': np.max(rr_intervals) - np.min(rr_intervals),
        'hr_mean': 60 / np.mean(rr_intervals),
        'hr_std': np.std(60 / rr_intervals),
    }
    return features

# Extract features from examples
peaks_normal, rr_normal = extract_rr_intervals(ecg_normal)
peaks_afib, rr_afib = extract_rr_intervals(ecg_afib)

features_normal = extract_hrv_features(rr_normal)
features_afib = extract_hrv_features(rr_afib)

print("Normal Sinus Rhythm Features:")
for k, v in features_normal.items():
    print(f"  {k}: {v:.3f}")

print("\nAtrial Fibrillation Features:")
for k, v in features_afib.items():
    print(f"  {k}: {v:.3f}")

if features_afib['rr_std'] > features_normal['rr_std']:
    print(f"\n💡 Notice: AFib has higher variability (rr_std {features_afib['rr_std']:.2f} vs {features_normal['rr_std']:.2f} s)")
else:
    print("\n💡 Notice: AFib does NOT show higher rr_std in this example")
if features_normal['rr_std'] > 0.1:
    print(f"   Caution: the 'normal' signal also looks variable (detected HR {features_normal['hr_mean']:.0f} bpm):")
    print("   the simple threshold peak detector misses or adds beats. Check peaks before trusting HRV features.")

Normal Sinus Rhythm Features:
  rr_mean: 1.048
  rr_std: 0.460
  rr_rmssd: 0.603
  rr_min: 0.456
  rr_max: 1.912
  rr_range: 1.456
  hr_mean: 57.252
  hr_std: 31.960

Atrial Fibrillation Features:
  rr_mean: 2.483
  rr_std: 1.116
  rr_rmssd: 2.210
  rr_min: 1.172
  rr_max: 3.900
  rr_range: 2.728
  hr_mean: 24.168
  hr_std: 15.102

💡 Notice: AFib has higher variability (rr_std 1.12 vs 0.46 s)
   Caution: the 'normal' signal also looks variable (detected HR 57 bpm vs 70 generated):
   the simple threshold peak detector misses or adds beats. Check peaks before trusting HRV features.


## 3. Generate Dataset and Train Classifier

Create a larger dataset and train Random Forest on HRV features. We keep the raw waveforms too, for the 1D CNN in Section 3b, and use the **same** train/test split for both models.

In [4]:
# Generate dataset
n_samples = 500
X_features = []
y_labels = []
X_raw = []  # raw waveforms for the 1D CNN (Section 3b)

print("Generating dataset...")
for i in range(n_samples):
    rhythm = 'normal' if i < n_samples // 2 else 'afib'
    label = 0 if rhythm == 'normal' else 1

    _, ecg = generate_synthetic_ecg(duration=30, rhythm=rhythm)
    peaks, rr = extract_rr_intervals(ecg)
    features = extract_hrv_features(rr)

    if len(features) > 0:
        X_features.append(list(features.values()))
        y_labels.append(label)
        X_raw.append(ecg)

X = np.array(X_features)
y = np.array(y_labels)
X_raw = np.array(X_raw)

feature_names = list(features_normal.keys())

print(f"Dataset shape: {X.shape}")
print(f"Features: {feature_names}")
print(f"Class distribution: {np.bincount(y)}")

# Split data (split indices, so the 1D CNN can reuse exactly the same split)
idx_train, idx_test = train_test_split(np.arange(len(y)), test_size=0.3, random_state=42, stratify=y)
X_train, X_test, y_train, y_test = X[idx_train], X[idx_test], y[idx_train], y[idx_test]

# Train Random Forest
import time
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
t0 = time.perf_counter()
rf.fit(X_train, y_train)
rf_train_time = time.perf_counter() - t0

# Evaluate
y_pred = rf.predict(X_test)
y_pred_proba = rf.predict_proba(X_test)[:, 1]

print("\nPerformance on Test Set:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'AFib']))

auroc = roc_auc_score(y_test, y_pred_proba)
print(f"AUROC: {auroc:.3f}")

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'AFib'], yticklabels=['Normal', 'AFib'])
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.title('Confusion Matrix: Feature-Based Classification')
plt.show()

Generating dataset...


Dataset shape: (500, 8)
Features: ['rr_mean', 'rr_std', 'rr_rmssd', 'rr_min', 'rr_max', 'rr_range', 'hr_mean', 'hr_std']
Class distribution: [250 250]

Performance on Test Set:
              precision    recall  f1-score   support

      Normal       1.00      1.00      1.00        75
        AFib       1.00      1.00      1.00        75

    accuracy                           1.00       150
   macro avg       1.00      1.00      1.00       150
weighted avg       1.00      1.00      1.00       150

AUROC: 1.000


## 3b. 1D CNN on the Raw ECG

The book (Section 7.5) describes a 1D CNN with convolutional layers of increasing width (**32 → 64 → 128 filters**), each followed by ReLU and max pooling, then **global average pooling** and dense layers. We train exactly that on the raw waveforms, using the same train/test split as the Random Forest, and compare:

- **Accuracy** (test accuracy, AUROC)
- **Interpretability** (feature importances vs. a black box)
- **Computational cost** (parameters, training time, inference time per 30 s ECG)

Design note: the input is downsampled to 125 Hz and the kernels are 15 samples wide, so the last layer "sees" about 2.4 s of signal: enough to compare 2–3 consecutive beats, which is what rhythm classification needs.

This section needs **TensorFlow** (Colab or local Jupyter). In the browser (JupyterLite) it is skipped and the feature-based classifier above is the result.

In [ ]:
try:
    import tensorflow as tf
    HAS_TF = True
except ImportError:
    HAS_TF = False
    print("ℹ️ This section needs TensorFlow (Colab or local Jupyter); it is skipped in the browser version.")

if HAS_TF:
    tf.random.set_seed(42)
    tf.keras.utils.set_random_seed(42)

    # Preprocess: downsample 250 -> 125 Hz, z-score each recording
    X_cnn = X_raw[:, ::2]
    X_cnn = (X_cnn - X_cnn.mean(axis=1, keepdims=True)) / (X_cnn.std(axis=1, keepdims=True) + 1e-8)
    X_cnn = X_cnn[..., np.newaxis].astype('float32')
    Xc_train, Xc_test = X_cnn[idx_train], X_cnn[idx_test]

    L = tf.keras.layers
    cnn = tf.keras.Sequential([
        L.Input(shape=X_cnn.shape[1:]),
        L.Conv1D(32, 15, padding='same', activation='relu'),  L.MaxPooling1D(4),
        L.Conv1D(64, 15, padding='same', activation='relu'),  L.MaxPooling1D(4),
        L.Conv1D(128, 15, padding='same', activation='relu'),
        L.GlobalAveragePooling1D(),
        L.Dense(32, activation='relu'),
        L.Dense(1, activation='sigmoid'),
    ])
    cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])

    t0 = time.perf_counter()
    history = cnn.fit(Xc_train, y_train, epochs=15, batch_size=32, validation_split=0.15, verbose=0)
    cnn_train_time = time.perf_counter() - t0

    cnn_proba = cnn.predict(Xc_test, verbose=0).ravel()
    cnn_pred = (cnn_proba >= 0.5).astype(int)
    print(f"1D CNN: {cnn.count_params():,} parameters, trained {len(history.history['loss'])} epochs in {cnn_train_time:.1f} s")
    print(f"Final training accuracy {history.history['accuracy'][-1]:.3f}, validation accuracy {history.history['val_accuracy'][-1]:.3f}")
    print("\nPerformance on Test Set:")
    print(classification_report(y_test, cnn_pred, target_names=['Normal', 'AFib']))
    print(f"AUROC: {roc_auc_score(y_test, cnn_proba):.3f}")


### Feature-Based vs. 1D CNN: Accuracy, Interpretability, Cost

Inference time is measured per single 30 s recording (as a wearable would process it), median of 50 runs. For the Random Forest it **includes** R-peak detection and HRV feature extraction; for the CNN it includes only the forward pass (preprocessing is negligible). The CNN's "size" is its trainable parameters; the forest's is its number of tree nodes (each node stores a feature index and threshold).

In [ ]:
from sklearn.metrics import accuracy_score

def median_ms(fn, n=50):
    times = []
    for i in range(n):
        t0 = time.perf_counter(); fn(i); times.append((time.perf_counter() - t0) * 1000)
    return np.median(times)

rf_acc = accuracy_score(y_test, y_pred)
rf_auc = roc_auc_score(y_test, y_pred_proba)
rf_nodes = sum(est.tree_.node_count for est in rf.estimators_)
rf_ms = median_ms(lambda i: rf.predict_proba(
    [list(extract_hrv_features(extract_rr_intervals(X_raw[idx_test[i % len(idx_test)]])[1]).values())]))

rows = [('HRV features + Random Forest', rf_acc, rf_auc, f'{rf_nodes:,} tree nodes', rf_train_time, rf_ms)]
if HAS_TF:
    cnn_acc = accuracy_score(y_test, cnn_pred)
    cnn_auc = roc_auc_score(y_test, cnn_proba)
    cnn_ms = median_ms(lambda i: cnn(Xc_test[i % len(Xc_test)][np.newaxis], training=False))
    rows.append(('1D CNN (raw ECG)', cnn_acc, cnn_auc, f'{cnn.count_params():,} parameters', cnn_train_time, cnn_ms))

print(f"{'Model':<30} {'Accuracy':>9} {'AUROC':>7} {'Size':>22} {'Train (s)':>10} {'Infer (ms/ECG)':>15}")
print("-" * 97)
for name, acc, auc_, size, tt, ms in rows:
    print(f"{name:<30} {acc:>9.3f} {auc_:>7.3f} {size:>22} {tt:>10.2f} {ms:>15.2f}")

# Interpretability: the forest tells us which features drove its decisions
imp = sorted(zip(feature_names, rf.feature_importances_), key=lambda x: -x[1])[:3]
print("\n🔍 Interpretability: top Random Forest features: " + ", ".join(f"{n} ({v:.2f})" for n, v in imp))
print("   The CNN has no built-in equivalent; it would need saliency methods (Chapter 10).")

if HAS_TF:
    diff = cnn_acc - rf_acc
    if abs(diff) < 0.01:
        verdict = "about as accurate as"
    elif diff > 0:
        verdict = f"{diff:.1%} MORE accurate than"
    else:
        verdict = f"{-diff:.1%} LESS accurate than"
    print(f"\n💡 On this synthetic data the 1D CNN is {verdict} the HRV-feature Random Forest,")
    print(f"   took {cnn_train_time / max(rf_train_time, 1e-9):.0f}× longer to train, and is "
          f"{cnn_ms / rf_ms:.1f}× {'slower' if cnn_ms > rf_ms else 'faster'} per ECG at inference.")
    if diff <= 0.01:
        print("   When the signal is rhythm irregularity, hand-crafted HRV features already capture it;")
        print("   the CNN's advantage appears with subtler morphology and much more training data.")
else:
    print("\nℹ️ CNN comparison skipped (no TensorFlow); the feature-based classifier is the browser result.")


## 4. The Prevalence Problem

Even with 98% specificity, screening low-risk populations yields poor PPV.

In [5]:
def calculate_ppv(sensitivity, specificity, prevalence):
    """Calculate positive predictive value."""
    tp = sensitivity * prevalence
    fp = (1 - specificity) * (1 - prevalence)
    ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
    fp_ratio = fp / tp if tp > 0 else float('inf')
    return ppv, fp_ratio

# From confusion matrix
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)

print(f"Model Performance (clinical setting):")
print(f"  Sensitivity: {sensitivity:.1%}")
print(f"  Specificity: {specificity:.1%}")

# Calculate PPV for different settings
settings = [
    ('Hospital (symptomatic)', 0.10),
    ('Yuki: 55 y, low-risk (0.5%)', 0.005),   # Chapter 3 base-rate example
    ('Low-risk, lower bound (0.1%)', 0.001)    # Chapter 3 range: 0.1-0.5%
]

print("\n" + "="*70)
print("POSITIVE PREDICTIVE VALUE BY SETTING")
print("="*70)
for setting, prev in settings:
    ppv, fp_ratio = calculate_ppv(sensitivity, specificity, prev)
    print(f"{setting:30s} | Prevalence: {prev:6.1%} | PPV: {ppv:6.1%} | FP ratio: {fp_ratio:5.1f}:1")
print("="*70)

if specificity == 1.0:
    print(f"\n⚠️  Our model made no false positives on {tn + fp} normal test ECGs, so PPV looks perfect.")
    print("   That reflects easy synthetic data and a small test set, not a real-world 0% false-alarm rate.")

# Illustrative wearable with realistic, imperfect performance
ill_sens, ill_spec = 0.95, 0.98
print(f"\nIllustrative device (sensitivity {ill_sens:.0%}, specificity {ill_spec:.0%}):")
for setting, prev in settings:
    ppv, fp_ratio = calculate_ppv(ill_sens, ill_spec, prev)
    print(f"{setting:30s} | Prevalence: {prev:6.1%} | PPV: {ppv:6.1%} | FP ratio: {fp_ratio:5.1f}:1")

ppv_yuki, fp_ratio_yuki = calculate_ppv(ill_sens, ill_spec, 0.005)
n_pop = 10_000
print(f"\nChapter 3 arithmetic for {n_pop:,} people like Yuki (prevalence 0.5%):")
print(f"   {n_pop*0.005:.0f} have AFib → {n_pop*0.005*ill_sens:.1f} detected; "
      f"{n_pop*0.995:,.0f} healthy → {n_pop*0.995*(1-ill_spec):.0f} false alarms")
print("\n⚠️  CLINICAL INSIGHT:")
print("   Same model, different settings → vastly different clinical utility")
print(f"   For Yuki (prevalence 0.5%), {ill_sens:.0%} sensitivity and {ill_spec:.0%} specificity → PPV {ppv_yuki:.0%}, "
      f"≈{fp_ratio_yuki:.0f} false alarms per true AFib ({1-ppv_yuki:.0%} of alerts are false)")

Model Performance (clinical setting):
  Sensitivity: 100.0%
  Specificity: 100.0%

POSITIVE PREDICTIVE VALUE BY SETTING
Hospital (symptomatic)         | Prevalence:  10.0% | PPV: 100.0% | FP ratio:   0.0:1
Screening (general)            | Prevalence:   0.5% | PPV: 100.0% | FP ratio:   0.0:1
Yuki (young, fit)              | Prevalence:   0.1% | PPV: 100.0% | FP ratio:   0.0:1

⚠️  Our model made no false positives on 75 normal test ECGs, so PPV looks perfect.
   That reflects easy synthetic data and a small test set, not a real-world 0% false-alarm rate.

Illustrative device (sensitivity 95%, specificity 98%):
Hospital (symptomatic)         | Prevalence:  10.0% | PPV:  84.1% | FP ratio:   0.2:1
Screening (general)            | Prevalence:   0.5% | PPV:  19.3% | FP ratio:   4.2:1
Yuki (young, fit)              | Prevalence:   0.1% | PPV:   4.5% | FP ratio:  21.0:1

⚠️  CLINICAL INSIGHT:
   Same model, different settings → vastly different clinical utility
   For Yuki (prevalence 0.1%), e

## 5. Motion Artifacts

Wearable devices are especially susceptible to motion artifacts during exercise.

In [6]:
def add_motion_artifact(ecg, intensity=0.5):
    """Add motion artifact to ECG signal."""
    artifact = intensity * np.random.randn(len(ecg))
    artifact = signal.filtfilt(*signal.butter(2, 0.05), artifact)
    return ecg + artifact

# Generate normal ECG with motion artifact
t_motion, ecg_clean = generate_synthetic_ecg(duration=10, rhythm='normal')
ecg_motion = add_motion_artifact(ecg_clean, intensity=0.3)

# Compare
fig, axes = plt.subplots(3, 1, figsize=(14, 10))

axes[0].plot(t_motion, ecg_clean, 'b-', linewidth=1)
axes[0].set_title('Clean ECG (Normal Sinus Rhythm)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Amplitude')
axes[0].grid(True, alpha=0.3)

axes[1].plot(t_motion, ecg_motion, 'orange', linewidth=1)
axes[1].set_title('ECG with Motion Artifact (e.g., during exercise)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Amplitude')
axes[1].grid(True, alpha=0.3)

axes[2].plot(t_afib, ecg_afib, 'r-', linewidth=1)
axes[2].set_title('True AFib', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Time (seconds)')
axes[2].set_ylabel('Amplitude')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Does our classifier call the motion-corrupted NORMAL ECG "AFib"?
motion_feats = extract_hrv_features(extract_rr_intervals(ecg_motion)[1])
if motion_feats:
    p_afib = rf.predict_proba([list(motion_feats.values())])[0, 1]
    print(f"Classifier on normal ECG + motion artifact: P(AFib) = {p_afib:.2f} → "
          f"{'FALSE ALARM' if p_afib >= 0.5 else 'correctly normal'}")
else:
    print("Classifier on normal ECG + motion artifact: too few peaks detected to classify")
print("⚠️  Observation: motion artifact can make the baseline look irregular, like AFib.")
print("   On real wearables this is a known source of false positives during exercise;")
print("   check above whether it fooled our classifier (one 10 s example, not a rate).")

Classifier on normal ECG + motion artifact: P(AFib) = 0.01 → correctly normal
⚠️  Observation: motion artifact can make the baseline look irregular, like AFib.
   On real wearables this is a known source of false positives during exercise;
   check above whether it fooled our classifier (one 10 s example, not a rate).


## 6. Key Takeaways

### What We Learned

1. **ECG rhythm classification**: Can use hand-crafted features (HRV) or deep learning (1D CNN). Section 3b printed which was more accurate and how much more compute the CNN needs on this synthetic data; with irregular R-R intervals as the main signal, simple HRV features are hard to beat

2. **Prevalence matters enormously**:
   - For the illustrative device in Section 4 (95% sensitivity, 98% specificity):
     Hospital (10% AFib): PPV ~84%; Yuki's setting (0.5% AFib): PPV ~19%, about 4 false alarms per true alert; 0.1% AFib: PPV ~4.5%
   - If our synthetic model scores (near-)perfectly on its small test set (Section 4), that hides this effect
   - Same model, vastly different utility

3. **Motion artifacts**: Major challenge for wearable devices
   - Exercise, arm movement mimics irregular rhythm
   - Leads to false positives

4. **Clinical implications**:
   - Wearable AFib detection useful for high-risk individuals
   - Should not be used for mass screening of low-risk populations
   - Need confirmatory testing pathways

5. **Yuki's experience** (Journey 2): Multiple false alarms caused anxiety, unnecessary testing, and ultimately loss of trust in technology

### Connections to Book Chapters

- **Chapter 3 (Seven Journeys)**: Yuki's story provides the clinical motivation
- **Chapter 5 (Evaluation)**: Prevalence dependence of PPV, metrics in imbalanced settings
- **Chapter 7 (Time Series)**: This chapter - signal processing, frequency domain features, 1D CNNs
- **Chapter 11 (Deployment)**: Deployment considerations, user experience, regulatory pathways

### Real-World Devices

- Apple Watch, Fitbit, KardiaMobile: FDA-cleared for AFib detection
- Clinical guidelines: Use with confirmatory testing, not as standalone diagnostic
- Appropriate for: Patients with symptoms or known AFib for monitoring

---

## Exercises

1. Implement frequency domain features (FFT, power spectral density)
2. Train the 1D CNN (Section 3b) with motion-artifact augmentation (Section 5) and compare its robustness with the HRV classifier
3. Simulate different prevalence settings and calculate required specificity for PPV >50%
4. Add varying levels of motion artifact and measure degradation

---

*This notebook is part of "AI in Healthcare" (Volume 1, Chapter 7: Time Series Analysis)*  
*This implements Journey 2 (Yuki - AFib Detection) from Chapter 3*  
*For clinical context, see Chapter 3. For time series methods, see Chapter 7.*